# Inertial-fusion ignition of a DT hot spot

*nuclear · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/nuclear/fusion_ignition)

A 0D model of the hot spot at the centre of an imploded deuterium–tritium capsule. Alpha particles from $\mathrm{D + T \to {}^4He + n}$ heat it, bremsstrahlung and expansion cool it, and heat escaping into the cold shell feeds the burn. A whole grid of starting temperatures and areal densities $(T_0, \rho R_0)$ is integrated as one batch: above a Lawson-type boundary the hot spot runs away and burns, below it quenches.

**Vector form**

$$3nT\,\frac{d\ln T}{dt} = f_\alpha(\rho R, T)\,\tfrac14 n^2\langle\sigma v\rangle E_\alpha - L\,\big(C_B n^2\sqrt T + 3p\,c_s/R\big),\qquad \frac{d\ln M}{dt} = \frac{P_{cond} + (1 - f_\alpha)P_\alpha}{3nT}$$

**Index (tensor) form**

$$P^{\alpha}_b = \tfrac14 n^2(1-\varphi_b)^2\langle\sigma v\rangle(T_b)\,E_\alpha,\qquad \dot y_{bk} = g_k(y_b),\quad y_{bk} = (\ln T_b,\ \ln\rho R_b,\ \varphi_b),\qquad \text{ignited}_b = \big[\max_t T_{tb} \ge 2\,T_{0b}\big]$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $(T_{0b}, \rho R_{0b})$ | `initial_state(t0, r0)` | (B, 3) | every grid point $b = i\,n_r + j$ is one hot spot; the state is $(\ln T, \ln\rho R, \varphi)$ |
| $\langle\sigma v\rangle(T)$ | `sigma_v(T)` | (B,) | Bosch–Hale DT reactivity, $\approx 1.14\times10^{-22}$ m³/s at 10 keV, peak $\approx 9\times10^{-22}$ near 65 keV |
| $f_\alpha$ | `alpha_fraction(rho_r, T)` | (B,) | share of the 3.5 MeV alpha energy stopped inside the hot spot |
| $C_B n^2\sqrt T$ | `C_BREMS * n**2 * sqrt(T)` | (B,) | bremsstrahlung, $C_B = 5.34\times10^{-37}$ W m³ keV$^{-1/2}$ |
| $3p\,c_s/R$ | `powers(...)["pdv"]` | (B,) | PdV work of a hot spot expanding at the sound speed |
| $P_{cond}$ | `3 * KAPPA0 * T**3.5 / R**2` | (B,) | Spitzer conduction into the shell; recycled as new hot mass (burn propagation) |
| $g_k(y_b)$ | `rhs(y, rho, loss, rho_r_max)` | (B, 3) | right-hand side for all hot spots at once, stepped with RK4 |
| $\rho R^*(T)$ | `balance_boundary(T, rho, loss)` | (n,) | Lawson-type curve where self-heating equals losses, by batched bisection |

Inertial confinement squeezes a few hundred micrograms of DT so fast that it burns before it flies apart. The hot spot ignites when the alpha particles it makes heat it faster than it cools. Two numbers decide this: the temperature, because $\langle\sigma v\rangle$ climbs steeply up to tens of keV, and the areal density $\rho R$, because a thicker hot spot stops its own alphas (alpha range roughly 0.3 g/cm² near 10 keV) and holds together longer against expansion. Bremsstrahlung alone sets the ideal ignition temperature, about 4.3 keV, where $\tfrac14\langle\sigma v\rangle E_\alpha = C_B\sqrt T$. The dashed curve is the instantaneous balance, the ICF form of the Lawson criterion; the shaded region is what the full integration finds, which also counts burn propagation into the cold shell. Raise the loss multiplier $L$ and the boundary moves to larger $\rho R$ and higher $T$. Research context: on 5 December 2022 the National Ignition Facility reported the first laboratory fusion experiment with target gain above 1, about 3.15 MJ of fusion energy from 2.05 MJ of laser light on target (approximate; Indirect Drive ICF Collaboration, Phys. Rev. Lett. 132, 065102, 2024), after reaching a burning plasma (Zylstra et al., Nature 601, 542, 2022) and a 1.3 MJ shot in August 2021 that exceeded the Lawson criterion for ignition (Phys. Rev. Lett. 129, 075001, 2022). Target gain counts only the laser energy reaching the target; the electricity used to run the lasers is far larger. Simplifications: a uniform 0D hot spot at fixed density (so the burn-up fraction is overestimated — nothing makes it disassemble except the PdV loss term), an illustrative alpha-deposition formula, approximate Spitzer conductivity ($\ln\Lambda \approx 10$), no radiation trapping, no mix of ablator material and equal ion and electron temperatures.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/nuclear/fusion_ignition/main.py
"""Main loop: grid of hot spots → power balance → batched RK4 burn → ignition map → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, fast_mode

from . import plot, solve, tensors
from .equation import confinement_time, ideal_ignition_temperature, number_density, sigma_v


def run(params: dict, out_dir: Path) -> Path:
    # CPU, float64: ⟨σv⟩ spans many decades and the burn is stiff.
    fast = fast_mode()
    rho, loss = params["density"], params["loss"]
    n = 10 if fast else params["n_grid"]
    t0, r0 = tensors.initial_grid(n, n, (params["t_min"], params["t_max_init"]), (0.05, 1.5))
    y0 = tensors.initial_state(t0, r0)  # (B, 3)

    for T in (5.0, 10.0, 20.0, 64.0):
        sv = sigma_v(torch.tensor(T, dtype=torch.float64)).item()
        print(f"⟨σv⟩_DT({T:4.0f} keV) = {sv:.3e} m³/s")
    print(f"ideal ignition temperature (alpha heating = bremsstrahlung): {ideal_ignition_temperature():.2f} keV")

    # Lawson-type boundary: where self-heating balances bremsstrahlung + PdV losses at t = 0
    t_curve = torch.logspace(0, 2, 300, dtype=torch.float64)
    r_curve = solve.balance_boundary(t_curve, rho, loss)
    ok = torch.isfinite(r_curve)
    k = int(torch.argmin(torch.where(ok, r_curve * t_curve, torch.full_like(t_curve, 1e9))))
    tau = confinement_time(t_curve[k], r_curve[k], rho).item()
    ntt = number_density(rho) * t_curve[k].item() * tau
    print(
        f"heating = losses curve (ρ = {rho:.0f} g/cm³, loss ×{loss}): min ρR·T = {r_curve[k] * t_curve[k]:.2f} "
        f"g cm⁻² keV at T = {t_curve[k]:.1f} keV, ρR = {r_curve[k]:.2f} g/cm²  →  n T τ = {ntt:.2e} keV s/m³ "
        f"(τ = R/3c_s = {tau * 1e12:.0f} ps)"
    )

    times, states = solve.integrate(
        y0, rho, loss, params["rho_r_max"], 40.0 if fast else params["t_end"], 0.25, 30 if fast else 120
    )
    ign = solve.ignited(states)
    print(f"\n{ign.sum().item()} of {len(ign)} hot spots ignite within {times[-1]:.0f} ps")
    first = solve.min_ignition_rho_r(t0, r0, ign)
    for i in range(0, len(t0), max(1, len(t0) // 6)):
        val = f"ρR₀ ≥ {first[i]:.2f} g/cm²" if torch.isfinite(first[i]) else "never (in this ρR range)"
        print(f"T₀ = {t0[i]:5.1f} keV: ignites for {val}")
    if ign.any():
        print(
            f"ignited hot spots: peak T up to {states[:, ign, 0].exp().max():.0f} keV, "
            f"mean final burn-up {100 * states[-1, ign, 2].mean():.0f} %"
        )

    rows = [int((t0 - target).abs().argmin()) for target in (6.0, 10.0)]
    cols = torch.linspace(0, len(r0) - 1, 5).round().long().tolist()  # five ρR₀ from smallest to largest
    shown = [i * len(r0) + j for i in rows for j in cols]
    return plot.animate(times, states, t0, r0, ign, t_curve, r_curve, shown, out_dir / "fusion_ignition.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/nuclear/fusion_ignition/equation.py
r"""Physics → tensor translation for a 0D inertial-fusion hot spot.

A hot spot of DT at temperature T, fixed density ρ and radius R (areal density ρR) with burn-up fraction φ.
Power densities (W/m³), with n = ρ/m̄ ions per m³ (half D, half T, m̄ = 2.5 u):

    P_α    = ¼ n² (1 − φ)² ⟨σv⟩(T) E_α,         E_α = 3.5 MeV          alpha heating (a fraction f_α is absorbed)
    P_br   = C_B n² √T,                        C_B = 5.34e-37 W m³ keV^-1/2   bremsstrahlung (Z = 1)
    P_pdV  = 3 p c_s / R,  p = 2 n T,  c_s = √(2T/m̄)                 expansion (PdV) work
    P_cond = 3 κ₀ T^{7/2} / R²                                        electron conduction into the cold shell

Energy balance of the hot spot, u = 3 n T (ions + electrons):

    u dT/dt / T = f_α P_α − L (P_br + P_pdV)                         L = loss multiplier
    (1/M) dM/dt = (P_cond + (1 − f_α) P_α) / u                        burn propagation: heat leaving the
                                                                      hot spot ablates cold fuel into it
    dφ/dt = ½ n (1 − φ)² ⟨σv⟩ − φ (1/M) dM/dt                        burn-up, diluted by fresh fuel

Conduction does not appear in the temperature equation: the heat it carries out is recycled as new hot
mass (the ablative hot-spot picture), so it grows ρR at fixed T instead of cooling the hot spot.
With fixed ρ, R ∝ M^{1/3}, so d ln(ρR)/dt = (1/3) d ln M/dt.

⟨σv⟩ is the Bosch–Hale (Nucl. Fusion 32, 611, 1992) parameterisation for D + T → α + n, valid 0.2–100 keV.
"""

import torch

KEV = 1.602176634e-16  # J
AMU = 1.66053907e-27  # kg
M_BAR = 2.5 * AMU  # mean DT ion mass
E_ALPHA = 3.5e3 * KEV  # J
E_FUSION = 17.6e3 * KEV  # J per reaction (alpha + neutron)
C_BREMS = 5.34e-37  # W m³ keV^-1/2
KAPPA0 = 1e14  # W m⁻¹ keV^-7/2: Spitzer electron conductivity with ln Λ ≈ 10 (approximate)
RHO_L_ALPHA = 0.3  # g/cm²: alpha range at 10 keV (illustrative)

# Bosch–Hale coefficients for T(d,n)α, ⟨σv⟩ in cm³/s with T in keV
BH = dict(
    BG=34.3827,
    mrc2=1124656.0,
    C1=1.17302e-9,
    C2=1.51361e-2,
    C3=7.51886e-2,
    C4=4.60643e-3,
    C5=1.35000e-2,
    C6=-1.06750e-4,
    C7=1.36600e-5,
)


def sigma_v(T: torch.Tensor) -> torch.Tensor:
    r"""⟨σv⟩(T) in m³/s (Bosch–Hale):  θ = T / [1 − T(C2 + T(C4 + T C6)) / (1 + T(C3 + T(C5 + T C7)))],
    ξ = (B_G²/4θ)^{1/3},  ⟨σv⟩ = C1 θ √(ξ / (m_r c² T³)) e^{−3ξ}."""
    c = BH
    T = T.clamp(0.2, 100.0)
    theta = T / (1 - T * (c["C2"] + T * (c["C4"] + T * c["C6"])) / (1 + T * (c["C3"] + T * (c["C5"] + T * c["C7"]))))
    xi = (c["BG"] ** 2 / (4 * theta)) ** (1 / 3)
    sv_cm3 = c["C1"] * theta * torch.sqrt(xi / (c["mrc2"] * T**3)) * torch.exp(-3 * xi)
    return sv_cm3 * 1e-6


def number_density(rho_gcc: float) -> float:
    """Ions per m³ for DT at density ρ in g/cm³."""
    return rho_gcc * 1e3 / M_BAR


def alpha_fraction(rho_r: torch.Tensor, T: torch.Tensor) -> torch.Tensor:
    r"""Fraction of alpha energy absorbed in the hot spot, f_α = 1 − exp(−ρR / ρℓ_α), ρℓ_α ∝ T^{5/4} (illustrative)."""
    range_alpha = RHO_L_ALPHA * (T / 10.0) ** 1.25
    return 1 - torch.exp(-rho_r / range_alpha)


def powers(T: torch.Tensor, rho_r: torch.Tensor, phi: torch.Tensor, rho_gcc: float) -> dict:
    """All power densities (W/m³) for a batch of hot spots; T in keV, ρR in g/cm², shapes (B,)."""
    n = number_density(rho_gcc)
    radius = rho_r * 10.0 / (rho_gcc * 1e3)  # ρR [g/cm²] → kg/m² (×10), divided by ρ [kg/m³] → m
    p_alpha = 0.25 * n**2 * (1 - phi) ** 2 * sigma_v(T) * E_ALPHA
    p_brems = C_BREMS * n**2 * torch.sqrt(T)
    sound = torch.sqrt(2 * T * KEV / M_BAR)
    p_pdv = 3 * (2 * n * T * KEV) * sound / radius
    p_cond = 3 * KAPPA0 * T**3.5 / radius**2
    return {"alpha": p_alpha, "brems": p_brems, "pdv": p_pdv, "cond": p_cond, "u": 3 * n * T * KEV}


def rhs(y: torch.Tensor, rho_gcc: float, loss: float, rho_r_max: float) -> torch.Tensor:
    r"""d/dt of the state y = (ln T, ln ρR, φ), shape (B, 3), in 1/s.

    ln T:   (f_α P_α − L(P_br + P_pdV)) / u
    ln ρR:  ⅓ (P_cond + (1 − f_α)P_α) / u × (1 − ρR/ρR_max)   (growth stops when the cold fuel is used up)
    φ:      ½ n (1 − φ)² ⟨σv⟩ − φ d ln M/dt
    """
    T, rho_r, phi = torch.exp(y[:, 0]), torch.exp(y[:, 1]), y[:, 2]
    p = powers(T, rho_r, phi, rho_gcc)
    f_a = alpha_fraction(rho_r, T)
    d_ln_t = (f_a * p["alpha"] - loss * (p["brems"] + p["pdv"])) / p["u"]
    d_ln_m = (p["cond"] + (1 - f_a) * p["alpha"]) / p["u"] * (1 - rho_r / rho_r_max).clamp(min=0.0)
    d_phi = 0.5 * number_density(rho_gcc) * (1 - phi) ** 2 * sigma_v(T) - phi * d_ln_m
    return torch.stack([d_ln_t, d_ln_m / 3, d_phi], dim=1)


def heating_balance(T: torch.Tensor, rho_r: torch.Tensor, rho_gcc: float, loss: float) -> torch.Tensor:
    r"""Initial self-heating margin f_α P_α − L(P_br + P_pdV) in W/m³ (> 0 ⇒ the hot spot heats up)."""
    p = powers(T, rho_r, torch.zeros_like(T), rho_gcc)
    return alpha_fraction(rho_r, T) * p["alpha"] - loss * (p["brems"] + p["pdv"])


def ideal_ignition_temperature() -> float:
    r"""T where alpha heating equals bremsstrahlung (f_α = 1, no other losses): ¼⟨σv⟩E_α = C_B √T."""
    T = torch.linspace(1.0, 20.0, 19001, dtype=torch.float64)
    margin = 0.25 * sigma_v(T) * E_ALPHA - C_BREMS * torch.sqrt(T)
    return T[int(torch.nonzero(margin > 0)[0])].item()


def confinement_time(T: torch.Tensor, rho_r: torch.Tensor, rho_gcc: float) -> torch.Tensor:
    """Expansion time R / (3 c_s) implied by the PdV term, in seconds."""
    radius = rho_r * 10.0 / (rho_gcc * 1e3)
    return radius / (3 * torch.sqrt(2 * T * KEV / M_BAR))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/nuclear/fusion_ignition/tensors.py
"""Build the concrete tensors: the grid of initial hot spots (T₀, ρR₀) stacked into one batch."""

import torch


def initial_grid(n_t: int, n_r: int, t_range: tuple, rho_r_range: tuple) -> tuple[torch.Tensor, torch.Tensor]:
    """Log-spaced starting temperatures T₀ (keV) and areal densities ρR₀ (g/cm²): (n_t,), (n_r,)."""
    t0 = torch.logspace(torch.log10(torch.tensor(t_range[0])), torch.log10(torch.tensor(t_range[1])), n_t)
    r0 = torch.logspace(torch.log10(torch.tensor(rho_r_range[0])), torch.log10(torch.tensor(rho_r_range[1])), n_r)
    return t0.double(), r0.double()


def initial_state(t0: torch.Tensor, r0: torch.Tensor) -> torch.Tensor:
    """y0 (B = n_t·n_r, 3): row b = (ln T₀, ln ρR₀, φ = 0) with b = i·n_r + j for T₀[i], ρR₀[j]."""
    T, R = torch.meshgrid(t0, r0, indexing="ij")
    return torch.stack([T.log().ravel(), R.log().ravel(), torch.zeros(T.numel(), dtype=torch.float64)], dim=1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/nuclear/fusion_ignition/solve.py
"""Integrate every hot spot at once with RK4 and classify ignition against quench."""

import torch

from problems._common.integrators import rk4_step
from problems._common.progress import steps

from .equation import heating_balance, rhs

T_FLOOR = 0.5  # keV: a hot spot this cold has quenched; we freeze it


def integrate(
    y0: torch.Tensor, rho_gcc: float, loss: float, rho_r_max: float, t_max_ps: float, dt_ps: float, n_save: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times in ps (S,), states (S, B, 3)) saved at n_save evenly spaced instants."""
    n_steps = int(round(t_max_ps / dt_ps))
    save_every = max(1, n_steps // (n_save - 1))
    dt = dt_ps * 1e-12

    def f(y, _t):
        d = rhs(y, rho_gcc, loss, rho_r_max)
        frozen = (y[:, 0] < torch.log(torch.tensor(T_FLOOR, dtype=y.dtype))) | (y[:, 2] > 0.99)
        return torch.where(frozen[:, None], torch.zeros_like(d), d)

    y, times, states = y0, [0.0], [y0]
    for k in steps(n_steps, "RK4 steps"):
        y = rk4_step(f, y, k * dt, dt)
        if (k + 1) % save_every == 0:
            times.append((k + 1) * dt_ps)
            states.append(y)
    return torch.tensor(times, dtype=torch.float64), torch.stack(states)


def ignited(states: torch.Tensor) -> torch.Tensor:
    """A hot spot ignites if it at least doubles its temperature (thermal runaway): (B,) bool."""
    ln_t = states[:, :, 0]
    return ln_t.max(dim=0).values >= ln_t[0] + torch.log(torch.tensor(2.0, dtype=ln_t.dtype))


def balance_boundary(T: torch.Tensor, rho_gcc: float, loss: float, n_iter: int = 60) -> torch.Tensor:
    """ρR*(T) where self-heating just balances the losses, by bisection in ln ρR for all T at once: (n,).

    Above ρR* the hot spot heats itself (alpha absorption rises and PdV loss falls with ρR); nan if
    even ρR = 30 g/cm² cannot balance bremsstrahlung (below the ideal ignition temperature).
    """
    lo = torch.full_like(T, -8.0)  # ln ρR
    hi = torch.full_like(T, 3.4)
    for _ in range(n_iter):
        mid = (lo + hi) / 2
        heats = heating_balance(T, mid.exp(), rho_gcc, loss) > 0
        hi, lo = torch.where(heats, mid, hi), torch.where(heats, lo, mid)
    ok = heating_balance(T, torch.full_like(T, 3.4).exp(), rho_gcc, loss) > 0
    return torch.where(ok, hi.exp(), torch.full_like(T, float("nan")))


def min_ignition_rho_r(t0: torch.Tensor, r0: torch.Tensor, ign: torch.Tensor) -> torch.Tensor:
    """For each starting temperature, the smallest ρR₀ that ignites (nan if none): (n_t,)."""
    grid = ign.view(len(t0), len(r0))
    first = torch.where(grid.any(1), grid.int().argmax(1), torch.zeros(len(t0), dtype=torch.long))
    return torch.where(grid.any(1), r0[first], torch.full_like(t0, float("nan")))


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/nuclear/fusion_ignition/plot.py
"""Animate the burn: ⟨σv⟩(T), trajectories in the (ρR, T) plane over the ignition map, and T(t)."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import ListedColormap
from matplotlib.ticker import FixedLocator, NullFormatter, NullLocator

from problems._common.plotting import frame_indices, save_gif

from .equation import sigma_v


def animate(
    times: torch.Tensor,
    states: torch.Tensor,
    t0: torch.Tensor,
    r0: torch.Tensor,
    ign: torch.Tensor,
    t_curve: torch.Tensor,
    r_curve: torch.Tensor,
    shown: list[int],
    path: Path,
) -> Path:
    tt = times.numpy()
    T = states[:, :, 0].exp().numpy()  # (S, B)
    R = states[:, :, 1].exp().numpy()
    ign_np = ign.numpy()

    fig, (ax_s, ax_p, ax_t) = plt.subplots(1, 3, figsize=(11.5, 3.9), gridspec_kw={"width_ratios": [0.8, 1.2, 1.0]})

    # (a) DT reactivity
    Tg = torch.logspace(0, 2, 200, dtype=torch.float64)
    ax_s.loglog(Tg.numpy(), sigma_v(Tg).numpy(), color="#2563eb", lw=2)
    sv10 = sigma_v(torch.tensor(10.0, dtype=torch.float64)).item()
    ax_s.plot([10], [sv10], "o", color="#dc2626")
    ax_s.annotate(f"{sv10:.2e} m³/s\nat 10 keV", (10, sv10), (12, 2e-24), fontsize=7, color="#b91c1c")
    ax_s.set_xlabel("T (keV)")
    ax_s.set_ylabel(r"$\langle\sigma v\rangle_{DT}$ (m³/s)")
    ax_s.set_title("Bosch–Hale reactivity", fontsize=9)

    # (b) ignition map in the (ρR, T) plane
    grid = ign_np.reshape(len(t0), len(r0)).astype(float)
    r_edges = _edges(r0.numpy())
    t_edges = _edges(t0.numpy())
    ax_p.pcolormesh(r_edges, t_edges, grid, cmap=ListedColormap(["#eff6ff", "#fee2e2"]), shading="flat", zorder=0)
    ax_p.plot(r_curve.numpy(), t_curve.numpy(), color="#374151", lw=1.5, ls="--", label="heating = losses at t = 0")
    ax_p.set_xscale("log")
    ax_p.set_yscale("log")
    ax_p.set_xlim(r_edges[0], max(r_edges[-1], R.max() * 1.1))
    ax_p.set_ylim(0.45, max(t_edges[-1], min(T.max(), 200) * 1.2))
    ax_p.set_xlabel("hot-spot areal density ρR (g/cm²)")
    ax_p.set_ylabel("T (keV)")
    ax_p.text(r_edges[-1] * 0.95, t_edges[0] * 1.1, "ignites", color="#b91c1c", ha="right", fontsize=8)
    ax_p.text(r_edges[0] * 1.1, t_edges[0] * 1.1, "quenches", color="#1e40af", fontsize=8)
    ax_p.axhline(0.5, color="#9ca3af", lw=1)
    colors = ["#dc2626" if ign_np[b] else "#2563eb" for b in shown]
    paths = [ax_p.plot([], [], lw=1.4, color=c)[0] for c in colors]
    dots = [ax_p.plot([], [], "o", ms=4, color=c)[0] for c in colors]
    ax_p.legend(loc="upper left", fontsize=7)
    title_p = ax_p.set_title("t = 0 ps", fontsize=9)

    # (c) temperature histories
    curves = [ax_t.plot([], [], lw=1.4, color=c)[0] for c in colors]
    ax_t.set_xlim(0, tt[-1])
    ax_t.set_yscale("log")
    ax_t.set_ylim(0.4, max(60.0, min(T[:, shown].max(), 200) * 1.3))
    ax_t.axhline(0.5, color="#9ca3af", lw=1)
    ax_t.set_xlabel("time (ps)")
    ax_t.set_ylabel("hot-spot T (keV)")
    burn = ax_t.text(0.03, 0.95, "", transform=ax_t.transAxes, va="top", fontsize=8)
    ax_t.set_title("red: ignition and burn, blue: quench", fontsize=9)

    # Plain tick labels on the log axes: mathtext labels would be re-typeset on every frame (slow).
    for ax, axis, ticks in (
        (ax_s, ax_s.xaxis, [1, 3, 10, 30, 100]),
        (ax_s, ax_s.yaxis, [1e-26, 1e-25, 1e-24, 1e-23, 1e-22, 1e-21]),
        (ax_p, ax_p.xaxis, [0.05, 0.1, 0.2, 0.5, 1, 2, 4]),
        (ax_p, ax_p.yaxis, [0.5, 1, 2, 5, 10, 20, 50, 100, 200]),
        (ax_t, ax_t.yaxis, [0.5, 1, 2, 5, 10, 20, 50, 100, 200]),
    ):
        lo, hi = ax.get_xlim() if axis is ax.xaxis else ax.get_ylim()
        kept = [v for v in ticks if lo <= v <= hi]
        axis.set_major_locator(FixedLocator(kept))
        axis.set_ticklabels([f"{v:g}" for v in kept])
        axis.set_minor_locator(NullLocator())
        axis.set_minor_formatter(NullFormatter())

    phi = states[:, :, 2].numpy()

    def update(s: int):
        for b, line, dot, curve in zip(shown, paths, dots, curves):
            line.set_data(R[: s + 1, b], T[: s + 1, b])
            dot.set_data([R[s, b]], [T[s, b]])
            curve.set_data(tt[: s + 1], T[: s + 1, b])
        title_p.set_text(f"{ign_np.sum()} of {len(ign_np)} hot spots ignite   t = {tt[s]:.0f} ps")
        burned = phi[s][ign_np].mean() if ign_np.any() else 0.0
        burn.set_text(f"mean burn-up of ignited hot spots: {100 * burned:.0f} %")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(tt), 48), path)


def _edges(centres: np.ndarray) -> np.ndarray:
    """Cell edges for a log-spaced grid of centres."""
    lc = np.log(centres)
    mids = (lc[1:] + lc[:-1]) / 2
    return np.exp(np.concatenate([[2 * lc[0] - mids[0]], mids, [2 * lc[-1] - mids[-1]]]))


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'density': 100.0,
    'loss': 1.0,
    'rho_r_max': 2.0,
    't_end': 200.0,
    'n_grid': 40,
    't_min': 2.0,
    't_max_init': 20.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.nuclear.fusion_ignition")]:
    del sys.modules[name]
main = importlib.import_module("problems.nuclear.fusion_ignition.main")
problem_dir = Path("problems/nuclear/fusion_ignition")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
